Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드다.

용도: 구글 클라우드(Google Cloud )에서 데이터 액세스 감사 로그가 꺼져 있어도, 클라우드 모니터링(Cloud Monitoring ) 지표를 기반으로 자격 증명(Credential )별 제미나이(Gemini ) 호출 추이를 실시간 분석하여 비용 급증 원인을 규명한다.

## 클라우드 모니터링 기반 제미나이 사용량 및 자격 증명 진단 (Gemini API Billing Spike Diagnostic )

### 1. 클라우드 모니터링 API 메트릭 수집

최근 90일간 발생한 시스템 메트릭 데이터를 수집한다. 감사 로그가 꺼져 있어도 수집 가능한 플랫폼 REST API를 사용하여 지표 데이터를 수집 진행한다.

In [ ]:
import datetime
import json
import os
import urllib.parse
import urllib.request
import google.auth
import google.auth.transport.requests

DAYS = 90
METRIC_FILTER = 'metric.type="serviceruntime.googleapis.com/api/request_count" AND (resource.labels.service="aiplatform.googleapis.com" OR resource.labels.service="generativelanguage.googleapis.com")'
RAW_METRIC_PATH = "tmp/gemini_raw_metrics.json"

try:
  creds, project_id = google.auth.default()
  if not project_id:
    raise ValueError("프로젝트 ID를 감지하지 못했다.")
  print(f"[성공] 활성화된 GCP 프로젝트 ID 감지: {project_id}")
except Exception as e:
  print("[오류] 활성화된 GCP 프로젝트 ID를 감지하지 못했다. google.auth 기본 자격 증명을 점검하기 바란다.")
  print("     (참고: GCP 콘솔(https://console.cloud.google.com/ )에서 직접 확인 가능하다.)")
  project_id = "your-project-id"

os.makedirs(os.path.dirname(RAW_METRIC_PATH), exist_ok=True)
print(f"프로젝트 '{project_id}'에서 최근 {DAYS}일간 발생한 시스템 메트릭 데이터를 수집하는 중이다...")

now = datetime.datetime.now(datetime.timezone.utc)
start_time = (now - datetime.timedelta(days=DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")
end_time = now.strftime("%Y-%m-%dT%H:%M:%SZ")

try:
  auth_req = google.auth.transport.requests.Request()
  creds.refresh(auth_req)
  access_token = creds.token

  query_params = {
    "filter": METRIC_FILTER,
    "interval.startTime": start_time,
    "interval.endTime": end_time
  }
  url = f"https://monitoring.googleapis.com/v3/projects/{project_id}/timeSeries?{urllib.parse.urlencode(query_params)}"

  req = urllib.request.Request(url)
  req.add_header("Authorization", f"Bearer {access_token}")

  with urllib.request.urlopen(req) as response:
    metric_data = json.loads(response.read().decode("utf-8"))

  with open(RAW_METRIC_PATH, "w", encoding="utf-8") as f:
    json.dump(metric_data, f, ensure_ascii=False, indent=2)

  if not metric_data.get("timeSeries"):
    print(f"[경고] 최근 {DAYS}일 동안 대상 API 호출 메트릭을 감지하지 못했다.")
    print("     (참고: 해당 기간에 메트릭 탐색기 지표 데이터가 아직 쌓이지 않았거나 호출 자체가 없었을 수 있다.)")
  else:
    print("[성공] 플랫폼 지표 데이터가 정상적으로 수집되어 로컬 파일에 적재되었다.")
except Exception as e:
  print(f"[오류] 지표 데이터를 수집하는 중 문제가 발생했다: {e}")

### 2. 파이썬 기반 메트릭 자격 증명 정밀 진단 실행

수집된 플랫폼 지표 데이터를 정밀 파싱하여, 자격 증명별 제미나이 호출 추이를 실시간 분석한다.

In [ ]:
import collections
import json
import os
import sys

def parse_metrics(file_path):
  if not os.path.exists(file_path):
    print(f"[오류] 메트릭 파일이 존재하지 않는다: {file_path}")
    return

  try:
    with open(file_path, "r", encoding="utf-8") as f:
      raw_data = json.load(f)
  except Exception as e:
    print(f"[오류] 메트릭 파일을 읽는 중 예외가 발생했다: {e}")
    return

  data = raw_data.get("timeSeries", [])
  if not data:
    print("[안내] 분석할 메트릭 데이터가 비어 있다.")
    return

  credential_counts = collections.defaultdict(int)
  method_counts = collections.defaultdict(int)
  service_counts = collections.defaultdict(int)

  for ts in data:
    metric = ts.get("metric", {})
    resource = ts.get("resource", {})
    labels = metric.get("labels", {})
    r_labels = resource.get("labels", {})

    cred_id = r_labels.get("credential_id") or labels.get("credential_id", "unknown-credential")
    service_name = r_labels.get("service") or labels.get("service", "unknown-service")
    method_name = r_labels.get("method") or labels.get("method", "unknown-method")

    points = ts.get("points", [])
    total_val = 0
    for pt in points:
      val_dict = pt.get("value", {})
      val_str = val_dict.get("int64Value", "0")
      try:
        total_val += int(val_str)
      except ValueError:
        pass

    credential_counts[cred_id] += total_val
    method_counts[method_name] += total_val
    service_counts[service_name] += total_val

  print("========================================================================")
  print("             클라우드 모니터링 기반 제미나이 사용량 진단 결과")
  print("========================================================================")
  print(f"매핑된 총 시계열 레코드: {len(data)}건")
  print()

  print("[1. API Services]")
  for service in sorted(service_counts.keys()):
    print(f"  - {service}: {service_counts[service]}회 호출")
  print()

  print("[2. Credentials and Keys]")
  for cred in sorted(credential_counts.keys()):
    print(f"  - {cred}: {credential_counts[cred]}회 호출")
  print()

  print("[3. Method Call Statistics]")
  for method in sorted(method_counts.keys()):
    print(f"  - {method}: {method_counts[method]}회 호출")
  print()
  print("========================================================================")

parse_metrics(RAW_METRIC_PATH)

### 3. 식별된 자격 증명 해석 방법

출력된 자격 증명 ID 결과를 기반으로 어떤 인증 수단에서 비용이 발생했는지 추적하는 가이드다.

* **'apikey:AIzaSy...' 형태인 경우**:
  구글 클라우드 콘솔의 'API 및 서비스 > 사용자 인증 정보'에서 일치하는 API 키를 찾아 삭제하거나 제한 조치한다.
  * 콘솔 링크: (https://console.cloud.google.com/apis/credentials )
* **'serviceAccount:...' 형태인 경우**:
  해당하는 서비스 계정이 사용 중인 프라이빗 키가 외부로 유출되었거나 비정상적인 백엔드 시스템에서 오호출 중인지 진단한다.
  * 콘솔 링크: (https://console.cloud.google.com/iam-admin/serviceaccounts )
* **'oauth2:...' 형태인 경우**:
  웹 앱 또는 클라이언트 애플리케이션에 발급된 OAuth 2.0 클라이언트 ID를 추적하여 호출 주체를 규명한다.
  * 콘솔 링크: (https://console.cloud.google.com/apis/credentials )